In [0]:
# STEP 1: Import Functions
from pyspark.sql.functions import col, avg, count, round, unix_timestamp

In [0]:
# STEP 2: Load Data from Silver
df_silver = spark.table("workspace.default.silver_tickets")

In [0]:
# STEP 3: Create Business Metrics (KPIs)
# I calculate the resolution time in hours for each ticket
df_with_metrics = df_silver.withColumn(
    "resolution_time_actual_hrs", 
    (unix_timestamp("resolved_at") - unix_timestamp("created_at")) / 3600
)

In [0]:
# STEP 4: Build the Gold Aggregated Table
# This is a 'Fact Table' summary for a Dashboard
df_gold_performance = df_with_metrics.groupBy("region", "product", "priority") \
    .agg(
        count("ticket_id").alias("total_tickets"),
        round(avg("resolution_time_actual_hrs"), 2).alias("avg_resolution_time"),
        round(avg("customer_satisfaction_score"), 2).alias("avg_satisfaction")
    ) \
    .orderBy("region", col("total_tickets").desc())

In [0]:
# STEP 5: Save to Gold Table
(df_gold_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_ticket_performance"))

print("Gold layer metrics generated successfully.")

Gold layer metrics generated successfully.
